In [1]:
import pandas as pd
import numpy as np

# Cargar el dataset (asegurate de que el archivo este en la misma carpeta o subido en Colab)
df = pd.read_csv('dirty_cafe_sales.csv')

# Ver las primeras 5 filas para confirmar que cargo bien
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
# 1. Ver si hay valores vacíos o nulos en Transaction ID
print("Nulos en Transaction ID:", df['Transaction ID'].isnull().sum())

Nulos en Transaction ID: 0


In [3]:
# 2. Ver si hay IDs duplicados
print("IDs duplicados:", df['Transaction ID'].duplicated().sum())

IDs duplicados: 0


In [4]:
# 3. Muestreo de valores únicos para detectar cosas raras
print("\nPrimeros 10 IDs únicos:")
print(df['Transaction ID'].unique()[:10])


Primeros 10 IDs únicos:
<StringArray>
['TXN_1961373', 'TXN_4977031', 'TXN_4271903', 'TXN_7034554', 'TXN_3160411',
 'TXN_2602893', 'TXN_4433211', 'TXN_6699534', 'TXN_4717867', 'TXN_2064365']
Length: 10, dtype: str


In [5]:
# A. Conteo total de registros en la columna
print(f"Total de filas analizadas: {len(df)}")
print("-" * 40)

# B. Conteo de frecuencias simples (frecuencia absoluta)
conteo = df['Item'].value_counts(dropna=False)

# C. Porcentajes (frecuencia relativa * 100)
porcentaje = df['Item'].value_counts(dropna=False, normalize=True) * 100

# D. Unir ambos en una sola tabla limpia
resumen_item = pd.DataFrame({
    'Cantidad': conteo,
    'Porcentaje (%)': porcentaje.round(2)
})

print(resumen_item)

Total de filas analizadas: 10000
----------------------------------------
          Cantidad  Porcentaje (%)
Item                              
Juice         1171           11.71
Coffee        1165           11.65
Salad         1148           11.48
Cake          1139           11.39
Sandwich      1131           11.31
Smoothie      1096           10.96
Cookie        1092           10.92
Tea           1089           10.89
UNKNOWN        344            3.44
NaN            333            3.33
ERROR          292            2.92


In [6]:
# 1. Convertir 'Price Per Unit' a numérico temporalmente para poder mapear
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')

# 2. Diccionario SOLO para precios con producto ÚNICO
mapa_precio_a_item = {
    1.0: 'Cookie',
    1.5: 'Tea',
    2.0: 'Coffee',
    5.0: 'Salad'
}

# 3. Limpiar las cadenas de texto sucias en 'Item' mandándolas a NaN
items_sucios = ['ERROR', 'UNKNOWN', 'Error', 'Unknown', 'Nan', 'None']
df['Item'] = df['Item'].replace(items_sucios, np.nan)

# 4. Imputar solo los NaN donde el precio sea único (1.0, 1.5, 2.0, 5.0)
df['Item'] = df['Item'].fillna(df['Price Per Unit'].map(mapa_precio_a_item))

# 5. Ver el resultado final de la columna Item con su tabla de porcentajes
conteo_item = df['Item'].value_counts(dropna=False)
porcentaje_item = df['Item'].value_counts(dropna=False, normalize=True) * 100

resumen_item_final = pd.DataFrame({
    'Cantidad': conteo_item,
    'Porcentaje (%)': porcentaje_item.round(2)
})

print(f"Total de registros: {len(df)}")
print("-" * 40)
print(resumen_item_final)

Total de registros: 10000
----------------------------------------
          Cantidad  Porcentaje (%)
Item                              
Coffee        1284           12.84
Salad         1270           12.70
Cookie        1209           12.09
Tea           1199           11.99
Juice         1171           11.71
Cake          1139           11.39
Sandwich      1131           11.31
Smoothie      1096           10.96
NaN            501            5.01


In [7]:
# A. Conteo total de registros en la columna
print(f"Total de filas analizadas: {len(df)}")
print("-" * 40)

# B. Conteo de frecuencias simples (frecuencia absoluta)
conteo = df['Item'].value_counts(dropna=False)

# C. Porcentajes (frecuencia relativa * 100)
porcentaje = df['Item'].value_counts(dropna=False, normalize=True) * 100

# D. Unir ambos en una sola tabla limpia
resumen_item = pd.DataFrame({
    'Cantidad': conteo,
    'Porcentaje (%)': porcentaje.round(2)
})

print(resumen_item)

Total de filas analizadas: 10000
----------------------------------------
          Cantidad  Porcentaje (%)
Item                              
Coffee        1284           12.84
Salad         1270           12.70
Cookie        1209           12.09
Tea           1199           11.99
Juice         1171           11.71
Cake          1139           11.39
Sandwich      1131           11.31
Smoothie      1096           10.96
NaN            501            5.01


In [8]:
# --- INSPECCIÓN DETALLADA DE QUANTITY ---

# A. Conteo de frecuencias (frecuencia absoluta)
conteo_qty = df['Quantity'].value_counts(dropna=False)

# B. Porcentajes (frecuencia relativa * 100)
porcentaje_qty = df['Quantity'].value_counts(dropna=False, normalize=True) * 100

# C. Tabla resumen
resumen_quantity = pd.DataFrame({
    'Cantidad': conteo_qty,
    'Porcentaje (%)': porcentaje_qty.round(2)
})

print(f"Total de registros: {len(df)}")
print("-" * 40)
print(resumen_quantity)

Total de registros: 10000
----------------------------------------
          Cantidad  Porcentaje (%)
Quantity                          
5             2013           20.13
2             1974           19.74
4             1863           18.63
3             1849           18.49
1             1822           18.22
UNKNOWN        171            1.71
ERROR          170            1.70
NaN            138            1.38


In [9]:
# Unificar textos de error a NaN real y convertir la columna a numérico
df['Quantity'] = pd.to_numeric(
    df['Quantity'].replace(['UNKNOWN', 'ERROR', 'Unknown', 'Error'], np.nan),
    errors='coerce'
)


In [10]:
# --- INSPECCIÓN DETALLADA DE QUANTITY ---

# A. Conteo de frecuencias (frecuencia absoluta)
conteo_qty = df['Quantity'].value_counts(dropna=False)

# B. Porcentajes (frecuencia relativa * 100)
porcentaje_qty = df['Quantity'].value_counts(dropna=False, normalize=True) * 100

# C. Tabla resumen
resumen_quantity = pd.DataFrame({
    'Cantidad': conteo_qty,
    'Porcentaje (%)': porcentaje_qty.round(2)
})

print(f"Total de registros: {len(df)}")
print("-" * 40)
print(resumen_quantity)

Total de registros: 10000
----------------------------------------
          Cantidad  Porcentaje (%)
Quantity                          
5.0           2013           20.13
2.0           1974           19.74
4.0           1863           18.63
3.0           1849           18.49
1.0           1822           18.22
NaN            479            4.79


In [11]:
# Unificar primero textos raros para inspección
precios_temp = df['Price Per Unit'].replace(['UNKNOWN', 'ERROR', 'Unknown', 'Error', 'Nan', 'None'], np.nan)

# Tabla de frecuencias y porcentajes
conteo_price = precios_temp.value_counts(dropna=False)
porcentaje_price = precios_temp.value_counts(dropna=False, normalize=True) * 100

resumen_price = pd.DataFrame({
    'Cantidad': conteo_price,
    'Porcentaje (%)': porcentaje_price.round(2)
})

print(f"Total de registros: {len(df)}")
print("-" * 40)
print(resumen_price)

Total de registros: 10000
----------------------------------------
                Cantidad  Porcentaje (%)
Price Per Unit                          
3.0                 2429           24.29
4.0                 2331           23.31
2.0                 1227           12.27
5.0                 1204           12.04
1.0                 1143           11.43
1.5                 1133           11.33
NaN                  533            5.33


In [12]:
# 1. Unificar temporalmente las cadenas de error a NaN para ver el desglose real
totales_brutos = df['Total Spent'].replace(['UNKNOWN', 'ERROR', 'Unknown', 'Error', 'Nan', 'None'], np.nan)

# 2. Conteo de frecuencias
conteo_total = totales_brutos.value_counts(dropna=False)

# 3. Porcentajes de cada valor
porcentaje_total = totales_brutos.value_counts(dropna=False, normalize=True) * 100

# 4. Crear tabla de resumen
resumen_total = pd.DataFrame({
    'Cantidad': conteo_total,
    'Porcentaje (%)': porcentaje_total.round(2)
})

print(f"Total de registros: {len(df)}")
print("-" * 40)
print(resumen_total)

Total de registros: 10000
----------------------------------------
             Cantidad  Porcentaje (%)
Total Spent                          
6.0               979            9.79
12.0              939            9.39
3.0               930            9.30
4.0               923            9.23
20.0              746            7.46
15.0              734            7.34
8.0               677            6.77
10.0              524            5.24
NaN               502            5.02
2.0               497            4.97
9.0               479            4.79
5.0               468            4.68
16.0              444            4.44
25.0              259            2.59
7.5               237            2.37
1.0               232            2.32
4.5               225            2.25
1.5               205            2.05


In [13]:
# Convertir temporalmente a numérico para evaluar
total_orig_num = pd.to_numeric(df['Total Spent'], errors='coerce')
qty_num = pd.to_numeric(df['Quantity'], errors='coerce')
price_num = pd.to_numeric(df['Price Per Unit'], errors='coerce')

# Total calculado teórico
total_calculado = qty_num * price_num

# 1. ¿Cuántos NaN/errores se pueden recuperar?
recuperables = total_orig_num.isna() & total_calculado.notna()

# 2. ¿Cuántos NaN se quedarán como NaN (porque falta Quantity o Price)?
sin_datos_suficientes = total_orig_num.isna() & total_calculado.isna()

print("--- DIAGNÓSTICO DE IMPUTACIÓN EN TOTAL SPENT ---")
print(f"Valores válidos ya existentes (se conservan intactos): {total_orig_num.notna().sum()}")
print(f"Valores NaN que SÍ se pueden llenar (Qty * Price)     : {recuperables.sum()}")
print(f"Valores NaN que SE QUEDAN en NaN (faltan factores)   : {sin_datos_suficientes.sum()}")

--- DIAGNÓSTICO DE IMPUTACIÓN EN TOTAL SPENT ---
Valores válidos ya existentes (se conservan intactos): 9498
Valores NaN que SÍ se pueden llenar (Qty * Price)     : 462
Valores NaN que SE QUEDAN en NaN (faltan factores)   : 40


In [14]:
# 1. Convertir textos de error ('ERROR', 'UNKNOWN', etc.) a NaN real en la columna original
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

# 2. Rellenar SOLO los vacíos (NaN) con la multiplicación de Quantity * Price Per Unit
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])

# 3. Comprobar resultado final
print("--- ESTADO FINAL DE TOTAL SPENT ---")
print(f"Nulos restantes en Total Spent: {df['Total Spent'].isna().sum()} ({(df['Total Spent'].isna().sum()/len(df))*100:.2f}%)")

--- ESTADO FINAL DE TOTAL SPENT ---
Nulos restantes en Total Spent: 40 (0.40%)


In [15]:
# 1. Reemplazar valores sucios habituales por NaN para lectura limpia
pagos_brutos = df['Payment Method'].replace(['UNKNOWN', 'ERROR', 'Unknown', 'Error', 'Nan', 'None'], np.nan)

# 2. Conteo de frecuencias e inclusión de nulos
conteo_pago = pagos_brutos.value_counts(dropna=False)
porcentaje_pago = pagos_brutos.value_counts(dropna=False, normalize=True) * 100

# 3. Construir la tabla resumen
resumen_pago = pd.DataFrame({
    'Cantidad': conteo_pago,
    'Porcentaje (%)': porcentaje_pago.round(2)
})

print(f"Total de registros: {len(df)}")
print("-" * 40)
print(resumen_pago)

Total de registros: 10000
----------------------------------------
                Cantidad  Porcentaje (%)
Payment Method                          
NaN                 3178           31.78
Digital Wallet      2291           22.91
Credit Card         2273           22.73
Cash                2258           22.58


In [16]:
# 1. Definir la lista de cadenas sucias o de error
errores_pago = ['ERROR', 'UNKNOWN', 'Error', 'Unknown', 'Nan', 'None', 'error', 'unknown']

# 2. Reemplazar permanentemente en la columna del DataFrame por NaN
df['Payment Method'] = df['Payment Method'].replace(errores_pago, np.nan)

# 3. Confirmar la distribución final fija
conteo_pago = df['Payment Method'].value_counts(dropna=False)
porcentaje_pago = df['Payment Method'].value_counts(dropna=False, normalize=True) * 100

resumen_pago_fijo = pd.DataFrame({
    'Cantidad': conteo_pago,
    'Porcentaje (%)': porcentaje_pago.round(2)
})

print("--- ESTADO FINAL DE PAYMENT METHOD ---")
print(resumen_pago_fijo)

--- ESTADO FINAL DE PAYMENT METHOD ---
                Cantidad  Porcentaje (%)
Payment Method                          
NaN                 3178           31.78
Digital Wallet      2291           22.91
Credit Card         2273           22.73
Cash                2258           22.58


In [17]:
import numpy as np
import pandas as pd

# 1. Limpiar temporalmente cadenas de error para ver el desglose real
ubicaciones_brutas = df['Location'].replace(['UNKNOWN', 'ERROR', 'Unknown', 'Error', 'Nan', 'None'], np.nan)

# 2. Conteo de frecuencias e inclusión de nulos
conteo_loc = ubicaciones_brutas.value_counts(dropna=False)
porcentaje_loc = ubicaciones_brutas.value_counts(dropna=False, normalize=True) * 100

# 3. Construir la tabla resumen
resumen_loc = pd.DataFrame({
    'Cantidad': conteo_loc,
    'Porcentaje (%)': porcentaje_loc.round(2)
})

print(f"Total de registros: {len(df)}")
print("-" * 40)
print(resumen_loc)

Total de registros: 10000
----------------------------------------
          Cantidad  Porcentaje (%)
Location                          
NaN           3961           39.61
Takeaway      3022           30.22
In-store      3017           30.17


In [18]:
# 1. Definir la lista de cadenas de error o faltantes habituales
errores_location = ['ERROR', 'UNKNOWN', 'Error', 'Unknown', 'Nan', 'None', 'error', 'unknown']

# 2. Reemplazar permanentemente las cadenas sucias por NaN
df['Location'] = df['Location'].replace(errores_location, np.nan)

# 3. Construir la tabla de distribución final
conteo_loc = df['Location'].value_counts(dropna=False)
porcentaje_loc = df['Location'].value_counts(dropna=False, normalize=True) * 100

resumen_loc_fijo = pd.DataFrame({
    'Cantidad': conteo_loc,
    'Porcentaje (%)': porcentaje_loc.round(2)
})

print("--- ESTADO FINAL DE LOCATION ---")
print(resumen_loc_fijo)

--- ESTADO FINAL DE LOCATION ---
          Cantidad  Porcentaje (%)
Location                          
NaN           3961           39.61
Takeaway      3022           30.22
In-store      3017           30.17


In [19]:

# 1. Detectar valores de error tipo texto y reemplazarlos por NaN
errores_fecha = ['ERROR', 'UNKNOWN', 'Error', 'Unknown', 'Nan', 'None', 'error', 'unknown']
df['Transaction Date'] = df['Transaction Date'].replace(errores_fecha, np.nan)

# 2. Información general del tipo de dato y nulos
print("--- INFORMACIÓN DE TRANSACTION DATE ---")
print(f"Tipo de dato actual: {df['Transaction Date'].dtype}")
print(f"Total de registros nulos/NaN: {df['Transaction Date'].isna().sum()} ({df['Transaction Date'].isna().mean()*100:.2f}%)")

# 3. Mostrar algunos valores no nulos de muestra para revisar el formato
print("\n--- MUESTRA DE VALORES ---")
print(df['Transaction Date'].dropna().head(10))

--- INFORMACIÓN DE TRANSACTION DATE ---
Tipo de dato actual: str
Total de registros nulos/NaN: 460 (4.60%)

--- MUESTRA DE VALORES ---
0    2023-09-08
1    2023-05-16
2    2023-07-19
3    2023-04-27
4    2023-06-11
5    2023-03-31
6    2023-10-06
7    2023-10-28
8    2023-07-28
9    2023-12-31
Name: Transaction Date, dtype: str


In [20]:
# 1. Convertir la columna a datetime
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], format='%Y-%m-%d', errors='coerce')

# 2. Verificar la conversión y el rango temporal
print("--- ESTADO FINAL DE TRANSACTION DATE ---")
print(f"Tipo de dato nuevo: {df['Transaction Date'].dtype}")
print(f"Total de registros nulos/NaT: {df['Transaction Date'].isna().sum()} ({df['Transaction Date'].isna().mean()*100:.2f}%)")
print(f"Fecha mínima: {df['Transaction Date'].min()}")
print(f"Fecha máxima: {df['Transaction Date'].max()}")

--- ESTADO FINAL DE TRANSACTION DATE ---
Tipo de dato nuevo: datetime64[us]
Total de registros nulos/NaT: 460 (4.60%)
Fecha mínima: 2023-01-01 00:00:00
Fecha máxima: 2023-12-31 00:00:00


In [21]:
# 1. Definir la lista completa de valores sucios a convertir a NaN
valores_sucios = ['ERROR', 'UNKNOWN', 'Error', 'Unknown', 'Nan', 'None', 'error', 'unknown']

# 2. Reemplazar cualquier texto sucio en todo el DataFrame por NaN
df = df.replace(valores_sucios, np.nan)

# 3. Columnas categóricas/texto a las que asignaremos 'N/A'
cols_texto = ['Item', 'Payment Method', 'Location']

for col in cols_texto:
    if col in df.columns:
        # Asegurar tipo string/categoría y rellenar nulos con 'N/A'
        df[col] = df[col].fillna('N/A').astype(str)

# 4. Verificación general de nulos y tipos de datos por columna
print("--- VERIFICACIÓN GENERAL DE VALORES FALTANTES (NaN / NaT) ---")
print(df.isna().sum())

print("\n--- MUESTRA DE CATEGORÍAS ÚNICAS POR COLUMNA ---")
for col in cols_texto:
    if col in df.columns:
        print(f"\nValores únicos en {col}:")
        print(df[col].unique())

--- VERIFICACIÓN GENERAL DE VALORES FALTANTES (NaN / NaT) ---
Transaction ID        0
Item                  0
Quantity            479
Price Per Unit      533
Total Spent          40
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

--- MUESTRA DE CATEGORÍAS ÚNICAS POR COLUMNA ---

Valores únicos en Item:
<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',      'N/A',
 'Sandwich',      'Tea',    'Juice']
Length: 9, dtype: str

Valores únicos en Payment Method:
<StringArray>
['Credit Card', 'Cash', 'N/A', 'Digital Wallet']
Length: 4, dtype: str

Valores únicos en Location:
<StringArray>
['Takeaway', 'In-store', 'N/A']
Length: 3, dtype: str


In [22]:
# Guardar el DataFrame limpio
df.to_csv('cafe_sales_cleaned.csv', index=False)